# S01 — SQL Foundations with DuckDB

**DuckDB** is an in-process analytical database. It runs entirely inside Python — no server, no setup — and queries pandas DataFrames directly. It is the standard in modern data stacks (dbt, MotherDuck, AWS Athena-compatible).

**Mental model:** DuckDB = PostgreSQL dialect + runs in memory + reads DataFrames/Parquet natively.

**Reference:** [DuckDB docs](https://duckdb.org/docs/)

**Topics:** SELECT mechanics, filtering, aggregation, JOINs, subqueries, CASE, string functions, date functions, NULL handling.

**Format:** Each exercise gives you a business question. Write a SQL query to answer it. The assert verifies your result against a pandas reference solution.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

# ── Load datasets ─────────────────────────────────────────────────────────────
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail['CustomerID'] = pd.to_numeric(retail['CustomerID'], errors='coerce')
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

# ── DuckDB connection — register DataFrames as virtual tables ─────────────────
con = duckdb.connect()
con.register('retail', retail)
con.register('credit', credit)

# Helper: run query and return DataFrame
def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

print("DuckDB ready. Tables: retail, credit")
q("SELECT COUNT(*) as n_rows, COUNT(DISTINCT Country) as n_countries FROM retail")

---
## Exercise 1 — SELECT, WHERE, ORDER BY

**Business question:** Find all UK transactions where Revenue exceeds £500, showing InvoiceNo, StockCode, Description, Quantity, UnitPrice, and Revenue. Sort by Revenue descending. Return top 20 rows.

**Concepts:** Column aliases, computed columns in SELECT, chained WHERE conditions, LIMIT.

In [ ]:
sql1 = """
-- YOUR SQL HERE
"""

result1 = q(sql1)
result1

In [ ]:
# --- ASSERTIONS ---
ref = (
    retail[(retail['Country'] == 'United Kingdom') & (retail['Revenue'] > 500)]
    [['InvoiceNo','StockCode','Description','Quantity','UnitPrice','Revenue']]
    .sort_values('Revenue', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
assert len(result1) == 20, f"Expected 20 rows, got {len(result1)}"
assert 'Revenue' in result1.columns
assert result1['Revenue'].iloc[0] >= result1['Revenue'].iloc[-1], "Must be sorted descending"
assert (result1['Revenue'] > 500).all(), "All rows must have Revenue > 500"
print(f"✓ Exercise 1 passed — {len(result1)} rows, max revenue: £{result1['Revenue'].max():,.2f}")

---
## Exercise 2 — Aggregation: GROUP BY, HAVING

**Business question:** For each country, compute total revenue, number of unique invoices, number of unique customers, and average order value (revenue / invoices). Only include countries with more than 100 unique customers. Sort by total revenue descending.

**Concepts:** COUNT(DISTINCT ...), SUM, AVG, HAVING vs WHERE, column aliasing in GROUP BY.

In [ ]:
sql2 = """
-- YOUR SQL HERE
"""

result2 = q(sql2)
result2

In [ ]:
# --- ASSERTIONS ---
ref = (
    retail.groupby('Country').agg(
        total_revenue=('Revenue','sum'),
        n_invoices=('InvoiceNo','nunique'),
        n_customers=('CustomerID','nunique')
    ).reset_index()
)
ref['avg_order_value'] = ref['total_revenue'] / ref['n_invoices']
ref = ref[ref['n_customers'] > 100].sort_values('total_revenue', ascending=False)

assert len(result2) == len(ref), f"Expected {len(ref)} rows, got {len(result2)}"
# All countries must have > 100 customers
cust_col = [c for c in result2.columns if 'cust' in c.lower()][0]
assert (result2[cust_col] > 100).all(), "HAVING filter not applied correctly"
rev_col = [c for c in result2.columns if 'rev' in c.lower()][0]
assert result2[rev_col].is_monotonic_decreasing, "Must be sorted by revenue descending"
print(f"✓ Exercise 2 passed — {len(result2)} qualifying countries")

---
## Exercise 3 — CASE WHEN: Conditional Logic

**Business question:** Classify each transaction into a revenue tier:
- `'Micro'` if Revenue < 10
- `'Small'` if Revenue between 10 and 99.99
- `'Medium'` if Revenue between 100 and 499.99
- `'Large'` if Revenue >= 500

Return a summary: tier, count of transactions, total revenue, % of total revenue. Sort by tier order (Micro → Small → Medium → Large).

**Concepts:** CASE WHEN, window function for % of total (SUM() OVER()), ordering custom sequences.

In [ ]:
sql3 = """
-- YOUR SQL HERE
"""

result3 = q(sql3)
result3

In [ ]:
# --- ASSERTIONS ---
assert len(result3) == 4, f"Expected 4 tiers, got {len(result3)}"
tier_col = [c for c in result3.columns if 'tier' in c.lower() or 'cat' in c.lower() or result3[c].dtype == object][0]
assert set(result3[tier_col]) == {'Micro','Small','Medium','Large'}
tier_order = ['Micro','Small','Medium','Large']
assert list(result3[tier_col]) == tier_order, "Must be sorted Micro→Small→Medium→Large"
# Check % of total sums to ~100
pct_col = [c for c in result3.columns if 'pct' in c.lower() or '%' in c or 'share' in c.lower()]
if pct_col:
    assert abs(result3[pct_col[0]].sum() - 100) < 0.1, "% must sum to 100"
print("✓ Exercise 3 passed")
print(result3.to_string(index=False))

---
## Exercise 4 — JOINs

**Business question:** For each customer, find their most recent invoice date and the total number of days between their first and last purchase. Then join with a per-customer revenue summary. Return customers who have been active for more than 30 days, with columns: CustomerID, first_purchase, last_purchase, active_days, total_revenue, n_orders.

**Concepts:** Self-join pattern using CTEs, DATEDIFF, INNER JOIN, derived tables.

**Hint:** Build two CTEs — one for date ranges, one for revenue — then join them.

In [ ]:
sql4 = """
-- YOUR SQL HERE
-- Use CTEs: WITH date_range AS (...), revenue_summary AS (...) SELECT ...
"""

result4 = q(sql4)
result4.head()

In [ ]:
# --- ASSERTIONS ---
date_range = retail.groupby('CustomerID')['InvoiceDate'].agg(['min','max']).reset_index()
date_range.columns = ['CustomerID','first_purchase','last_purchase']
date_range['active_days'] = (date_range['last_purchase'] - date_range['first_purchase']).dt.days
rev_sum = retail.groupby('CustomerID').agg(total_revenue=('Revenue','sum'), n_orders=('InvoiceNo','nunique')).reset_index()
ref = date_range.merge(rev_sum, on='CustomerID').query('active_days > 30')

assert len(result4) == len(ref), f"Expected {len(ref)} rows, got {len(result4)}"
active_col = [c for c in result4.columns if 'day' in c.lower() or 'active' in c.lower()][0]
assert (result4[active_col] > 30).all(), "All customers must have active_days > 30"
assert 'CustomerID' in result4.columns or 'customerid' in result4.columns.str.lower().tolist()
print(f"✓ Exercise 4 passed — {len(result4)} customers active > 30 days")

---
## Exercise 5 — Subqueries & EXISTS

**Business question:** Find all products (StockCode + Description) that were purchased in EVERY country in the top-5 countries by revenue. Use a subquery or EXISTS pattern — not a pivot.

**Concepts:** Subqueries in WHERE, IN with subquery, correlated subqueries, set logic.

**Hint:** First find the top-5 countries. Then find products where COUNT(DISTINCT Country) = 5 within those countries.

In [ ]:
sql5 = """
-- YOUR SQL HERE
"""

result5 = q(sql5)
result5.head(10)

In [ ]:
# --- ASSERTIONS ---
top5 = retail.groupby('Country')['Revenue'].sum().nlargest(5).index.tolist()
filtered = retail[retail['Country'].isin(top5)]
ref = (
    filtered.groupby(['StockCode','Description'])['Country']
    .nunique().reset_index()
    .query('Country == 5')[['StockCode','Description']]
)
assert len(result5) == len(ref), f"Expected {len(ref)} products, got {len(result5)}"
assert 'StockCode' in result5.columns or 'stockcode' in result5.columns.str.lower().tolist()
print(f"✓ Exercise 5 passed — {len(result5)} products sold in all top-5 countries")

---
## Exercise 6 — Date Functions

**Business question:** Build a monthly sales calendar. For each year-month, compute: total revenue, MoM revenue change (£), MoM revenue change (%), and a `trend` flag: `'Up'`, `'Down'`, or `'Flat'` (< 1% change).

**Concepts:** DATE_TRUNC, STRFTIME, LAG() window function, ROUND, CASE on computed columns.

In [ ]:
sql6 = """
-- YOUR SQL HERE
-- Hint: use DATE_TRUNC('month', InvoiceDate) to group by month
-- Use LAG(total_revenue) OVER (ORDER BY month) for previous period
"""

result6 = q(sql6)
result6

In [ ]:
# --- ASSERTIONS ---
assert len(result6) > 0
trend_col = [c for c in result6.columns if 'trend' in c.lower()][0]
assert set(result6[trend_col].dropna()).issubset({'Up','Down','Flat'})
mom_pct_col = [c for c in result6.columns if 'pct' in c.lower() or 'change' in c.lower() or '%' in c]
assert len(mom_pct_col) >= 1, "Must have a MoM % change column"
# First row LAG should be NULL
first_mom = result6[mom_pct_col[0]].iloc[0]
assert pd.isna(first_mom) or first_mom == 0, "First row MoM should be NULL or 0"
print("✓ Exercise 6 passed")
print(result6.to_string(index=False))

---
## Exercise 7 — String Functions

**Business question:** Clean and categorize product descriptions. From the retail table:
1. Strip leading/trailing whitespace from Description
2. Convert to title case
3. Extract the first word of each description as `product_family`
4. Flag descriptions containing the word `'SET'` (case-insensitive) as `is_set = TRUE`
5. Count products per `product_family` where `is_set = TRUE` — return top 10 families.

**Concepts:** TRIM, UPPER/LOWER, SPLIT_PART, LIKE/ILIKE, REGEXP_MATCHES, string aggregation.

In [ ]:
sql7 = """
-- YOUR SQL HERE
"""

result7 = q(sql7)
result7

In [ ]:
# --- ASSERTIONS ---
assert len(result7) == 10, f"Expected top 10 families, got {len(result7)}"
family_col = [c for c in result7.columns if 'family' in c.lower() or 'word' in c.lower() or result7[c].dtype == object][0]
count_col = [c for c in result7.columns if 'count' in c.lower() or 'n_' in c.lower() or result7[c].dtype in [np.int64, np.float64]][0]
assert result7[count_col].is_monotonic_decreasing, "Must be sorted by count descending"
# All families should be single words
assert result7[family_col].str.contains(' ').sum() == 0, "product_family must be single words"
print("✓ Exercise 7 passed")
print(result7.to_string(index=False))

---
## Exercise 8 — NULL Handling

**Business question:** Audit data quality. For each column in the retail table, return: column_name, total_rows, null_count, null_pct, and a `quality_flag`:
- `'OK'` if null_pct = 0
- `'Warning'` if null_pct between 0 and 5%
- `'Critical'` if null_pct > 5%

**Concepts:** INFORMATION_SCHEMA, dynamic SQL concepts, COALESCE, NULLIF, COUNT(*) vs COUNT(col).

**Hint:** DuckDB supports `UNPIVOT` or you can use `UNION ALL` to stack per-column counts.

In [ ]:
sql8 = """
-- YOUR SQL HERE
-- Hint: build one row per column using UNION ALL or DuckDB's UNPIVOT
"""

result8 = q(sql8)
result8

In [ ]:
# --- ASSERTIONS ---
assert len(result8) == len(retail.columns), f"Expected {len(retail.columns)} rows (one per column)"
flag_col = [c for c in result8.columns if 'flag' in c.lower() or 'quality' in c.lower()][0]
assert set(result8[flag_col]).issubset({'OK','Warning','Critical'})
null_col = [c for c in result8.columns if 'null_count' in c.lower() or 'null_c' in c.lower()][0]
assert (result8[null_col] >= 0).all()
print("✓ Exercise 8 passed")
print(result8.to_string(index=False))

---
## Exercise 9 — Multi-table JOIN Chain

**Business question:** Create an enriched customer profile by joining two derived tables:
1. Customer revenue summary (from retail): CustomerID, total_revenue, n_orders, first_order, last_order
2. A synthetic customer attributes table (generated below): CustomerID, segment, region, account_manager

Return all customers from the revenue table, with segment/region/account_manager where available (LEFT JOIN). Add a column `has_profile = TRUE/FALSE`.
Filter to customers with total_revenue > 500. Sort by total_revenue desc.

**Concepts:** LEFT JOIN, COALESCE for null handling, IS NOT NULL filtering, multi-step CTEs.

In [ ]:
# Build synthetic customer attributes table
np.random.seed(42)
customer_ids = retail['CustomerID'].dropna().unique()
sample_ids = np.random.choice(customer_ids, size=int(len(customer_ids) * 0.7), replace=False)
customer_attrs = pd.DataFrame({
    'CustomerID': sample_ids,
    'segment': np.random.choice(['Enterprise','SMB','Consumer'], len(sample_ids)),
    'region': np.random.choice(['North','South','East','West'], len(sample_ids)),
    'account_manager': np.random.choice(['Alice','Bob','Carol','Dave'], len(sample_ids))
})
con.register('customer_attrs', customer_attrs)

sql9 = """
-- YOUR SQL HERE
"""

result9 = q(sql9)
result9.head()

In [ ]:
# --- ASSERTIONS ---
rev_summary = retail.groupby('CustomerID').agg(total_revenue=('Revenue','sum')).reset_index()
ref_count = len(rev_summary[rev_summary['total_revenue'] > 500])
assert len(result9) == ref_count, f"Expected {ref_count} rows, got {len(result9)}"
assert 'has_profile' in result9.columns or 'has_profile' in result9.columns.str.lower().tolist()
rev_col = [c for c in result9.columns if 'rev' in c.lower()][0]
assert (result9[rev_col] > 500).all()
assert result9[rev_col].is_monotonic_decreasing
print(f"✓ Exercise 9 passed — {len(result9)} customers, {result9['has_profile'].sum()} with profiles")

---
## Exercise 10 — Capstone: Business Intelligence Query

**Business question:** Write a single SQL query (using CTEs) that produces a complete customer health scorecard:

For each customer with at least 2 orders:
- `CustomerID`
- `total_revenue`, `n_orders`, `avg_order_value`
- `days_since_last_order`: days between their last order and the dataset's max date
- `order_frequency_days`: avg days between consecutive orders (total_active_days / (n_orders - 1))
- `revenue_trend`: compare last 3 months revenue to prior 3 months — `'Growing'`, `'Declining'`, `'Stable'` (< 10% diff)
- `health_score`: integer 1–5 computed as:
  - +2 if days_since_last_order < 30
  - +1 if days_since_last_order between 30–90
  - +2 if n_orders >= 5
  - +1 if n_orders between 2–4
  - capped at 5

Sort by health_score desc, total_revenue desc.

In [ ]:
sql10 = """
-- YOUR SQL HERE — use multiple CTEs
"""

result10 = q(sql10)
result10.head(10)

In [ ]:
# --- ASSERTIONS ---
assert len(result10) > 0
required = ['CustomerID', 'total_revenue', 'n_orders', 'avg_order_value',
            'days_since_last_order', 'revenue_trend', 'health_score']
result_cols_lower = result10.columns.str.lower().tolist()
for col in required:
    assert col.lower() in result_cols_lower, f"Missing column: {col}"

order_col = [c for c in result10.columns if 'n_order' in c.lower() or 'orders' in c.lower()][0]
assert (result10[order_col] >= 2).all(), "Must have at least 2 orders"

trend_col = [c for c in result10.columns if 'trend' in c.lower()][0]
assert set(result10[trend_col].dropna()).issubset({'Growing','Declining','Stable'})

score_col = [c for c in result10.columns if 'health' in c.lower() or 'score' in c.lower()][0]
assert result10[score_col].between(1, 5).all(), "health_score must be 1–5"

print(f"✓ Exercise 10 passed — {len(result10)} customers scored")
print(result10[[order_col, trend_col, score_col]].value_counts().head())